In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:


customers_data = [
    (1, "Alpha Corp", "Retail", "US", "2022-01-10"),
    (2, "Beta LLC", "Manufacturing", "US", "2022-03-15"),
    (3, "Gamma Ltd", "Finance", "UK", "2022-05-20"),
    (4, "Delta Inc", "Retail", None, "2022-06-01"),
    (5, "Alpha Corp", "Retail", "US", "2022-01-10"),  # duplicate
    (6, "Epsilon GmbH", "Manufacturing", "DE", "2022-07-12")
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "company_name", "industry", "country", "created_date"]
)




In [0]:
customers_df.count()

In [0]:
display(customers_df.distinct())

In [0]:
customers_df.dropDuplicates().count()

In [0]:
%sql
create catalog shop;
create schema shop.b2b;


In [0]:
customers_df.write.format("delta").saveAsTable("shop.b2b.customer")

In [0]:
%sql
select * from shop.b2b.customer

In [0]:
df=spark.read.table("shop.b2b.customer")

In [0]:
display(df)

In [0]:
orders_data = [
    (101, 1, "2023-01-05", 5000, "completed"),
    (102, 1, "2023-01-20", 3000, "completed"),
    (103, 2, "2023-02-10", 7000, "cancelled"),
    (104, 3, "2023-03-15", 10000, "completed"),
    (105, 4, "2023-04-01", None, "completed"),
    (106, 7, "2023-04-10", 4000, "completed"),  # no customer
    (107, 2, "2023-02-10", 7000, "cancelled"),  # duplicate
    (108, 6, "2023-05-01", 8000, "completed")
]

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "order_date", "amount", "status"]
)

orders_df.write.format("delta").saveAsTable("shop.b2b.orders")
orders_df.show()


In [0]:
%sql
select * from shop.b2b.orders

In [0]:
orders_df = orders_df.withColumn("amount", coalesce(col("amount"), lit(0)))

In [0]:
orders_df.fillna({'amount':0})

In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable("shop.b2b.orders")


In [0]:
display(orders_df)

In [0]:
%sql
select o.customer_id,c.company_name, sum(o.amount) as total_revenue from shop.b2b.customer c left join shop.b2b.orders o on c.customer_id=o.customer_id
where o.status='completed' 
group by o.customer_id,c.company_name
 order by o.customer_id

In [0]:
%sql
with no_shop_customer as (
  select c.customer_id, c.company_name,o.order_id from shop.b2b.customer c left join shop.b2b.orders o on c.customer_id=o.customer_id )
select customer_id, company_name from no_shop_customer where order_id is Null

In [0]:
%sql
select * from shop.b2b.orders o left anti join shop.b2b.customer c on o.customer_id=c.customer_id

In [0]:
%sql
with customer_dupe as (
    select c.*, row_number() over(partition by c.company_name, c.created_date order by c.customer_id ) as r_nm from shop.b2b.customer c
)
select count(*) from customer_dupe where r_nm=2

In [0]:
from pyspark.sql.window import Window

dfc=customers_df.withColumn('rn',row_number().over(Window.partitionBy('company_name','created_date').orderBy('created_date'))).filter(col("rn")==1).drop('rn')
display(dfc)

In [0]:
dfc.write.format('delta').mode('overwrite').saveAsTable('shop.b2b.customer')

In [0]:
%sql
describe history shop.b2b.customer

In [0]:
%sql
describe extended shop.b2b.customer

In [0]:
%sql
with group_customers as (
  select customer_id, sum(amount) as total_revenue from shop.b2b.orders where status='completed' group by customer_id
) ,
rank_customers as (
 select g.*, dense_rank() over(order by total_revenue desc) as rn,total_revenue from group_customers g
)
select c.customer_id, c.company_name,r.total_revenue from shop.b2b.customer c inner join rank_customers r on c.customer_id=r.customer_id where r.rn < 3

In [0]:
%sql
select * from shop.b2b.customer

In [0]:
dfc1=dfc.withColumn('industry',coalesce(col('industry'),lit("Unknown"))).withColumn('country',coalesce(col("country"),lit("Unknown")))

dfc1.write.format("delta").mode("overwrite").saveAsTable("shop.b2b.customer")

In [0]:
%sql
select * from shop.b2b.orders

In [0]:
%sql
-- For each customer:Show current order, Show previous order amount, Order by order_date
select customer_id, order_id, LAG(amount,1,0) over(partition by customer_id order by order_date) as previopus_amount_order from shop.b2b.orders 


In [0]:
%sql
-- For each customer:Calculate number of days between current order and previous order, First order should return NULL
select customer_id, order_date, cast(order_date as date) - lag(cast(order_date as date),1) over(partition by customer_id order by cast(order_date as date) ) from shop.b2b.orders

In [0]:
%sql
-- first and last order date per customer
select customer_id, first_value(cast(order_date as date)) over(partition by customer_id order by cast(order_date as date)) as first_date, last_value(cast(order_date as date)) over(partition by customer_id order by cast(order_date as date) rows between unbounded preceding and unbounded following ) as last_ordered from shop.b2b.orders 

In [0]:
%sql
select o.* , sum(amount) over(order by order_date) as running_total from shop.b2b.orders o

In [0]:
%sql
select o.* , sum(amount) over(order by cast(order_date as date) range between interval '6' day preceding and current row) as avg_rolling_sum from shop.b2b.orders o

In [0]:
Q7: Detect revenue drop

For each customer:

Compare current order amount with previous order

Flag 'DROP' if current < previous, else 'OK'

In [0]:
df=spark.read.table("shop.b2b.orders")
display(df)

In [0]:
df.